# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [16]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [17]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'capabilities page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'}]}

In [18]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [20]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'proficient page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'article',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/'},
  {'type': 'article',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'article',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'article',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'}]}

In [21]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learning resources', 'url': 'https://huggingface.co/learn'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'endpoints page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'Discord invite', 'url': 'https://huggingface.co/join/discord'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [22]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [23]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
11 days ago
•
250k
•
747
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
1 day ago
•
129k
•
1k
mistralai/Mistral-Small-4-119B-2603
Updated
5 days ago
•
9.86k
•
279
fishaudio/s2-pro
Updated
11 days ago
•
11.7k
•
696
baidu/Qianfan-OCR
Updated
3 days ago
•
4.32k
•
273
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.44k
Wan2.2 14B Preview
🐌
1.44k
generate a video from an image with a text prompt
Running
on
Zero
MCP
Featured
385
FireRed Image Edit 1.0 Fast
🌖

In [31]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [25]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [26]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 21 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nHauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive\nUpdated\n11 days ago\n•\n250k\n•\n747\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\n1 day ago\n•\n129k\n•\n1k\nmistralai/Mistral-Small-4-119B-2603\nUpdated\n5 days ago\n•\n9.86k\n•\n279\nfishaudio/s2-pro\nUpdated\n11 days ago\n•\n11.7k\n•\n696\nbaidu/Qianfan-OCR\nUpdated\n3 days ago\n•\n4.32k\n•\n273\nBrowse 2

In [27]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [28]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face Brochure

---

## Who We Are  
Hugging Face is the vibrant AI community building the future of machine learning. Acting as a global hub, our platform enables researchers, developers, and enterprises to create, share, and collaborate on AI models, datasets, and applications — accelerating innovation in machine learning across industries.  

---

## What We Offer  

- **Extensive AI Model Repository**  
  Explore over 2 million machine learning models spanning multiple modalities including text, image, video, audio, and even 3D. With trending and recently updated models from community leaders and industry pioneers, Hugging Face is your go-to source for state-of-the-art AI capabilities.

- **Massive Dataset Library**  
  Access and contribute to over 500,000 datasets, including specialized collections in healthcare robotics, natural language processing, computer vision, and reinforcement learning.  

- **Spaces – Collaborative AI Apps**  
  Build and deploy unlimited public AI applications directly on the platform. Engage with cutting-edge projects like text-to-video generators, image editing tools, and more while showcasing your creations.  

- **Open Source Stack for Faster Development**  
  Utilize Hugging Face’s open source tools and libraries to accelerate your machine learning workflows from experimentation to production.  

- **Enterprise Solutions**  
  Tailored AI platforms and support for businesses aiming to integrate advanced ML technologies at scale while ensuring security and compliance.  

---

## Our Community & Culture  

At Hugging Face, collaboration is at our core. We foster an open, inclusive environment where enthusiasts and experts from across the globe contribute to a shared mission of democratizing AI. Our culture encourages continuous learning, innovation, and transparency — empowering you to build your portfolio and establish your presence in the ML space.  

The community is supported with resources such as documentation, community blogs, and case studies highlighting real-world impact and ethical AI developments.  

---

## Who Uses Hugging Face?  

Our users are as diverse as AI itself, including:  
- **Researchers** pushing the boundaries of NLP, computer vision, audio processing, and robotics.  
- **Developers** building innovative AI applications and tools.  
- **Enterprises** leveraging open source and custom solutions to transform their business operations.  
- **AI enthusiasts and students** learning and sharing models to grow professionally.  

From startups to industry leaders, Hugging Face serves millions of users worldwide who rely on our platform for collaboration and AI acceleration.  

---

## Careers at Hugging Face  

We are continuously seeking passionate individuals eager to impact the AI world. Open positions typically include roles in:  
- Machine Learning Research and Engineering  
- Software Development and Infrastructure  
- Community Management and Developer Advocacy  
- Product Management and Design  

Join a team that values innovation, openness, and community — work alongside top talent dedicated to shaping the future of AI.  

For current opportunities, visit our Careers page and become part of the AI revolution.  

---

## Get Started  

- **Explore AI Models and Datasets:** Dive into 2M+ AI models and 500k+ datasets waiting to power your projects.  
- **Create & Share:** Build your own Spaces and share applications with a global audience.  
- **Join the Community:** Engage with fellow ML practitioners through blogs, case studies, and collaboration.  

Sign up today at [huggingface.co](https://huggingface.co) and start accelerating your machine learning journey!  

---

*Hugging Face – The AI community building the future.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [29]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [30]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is a pioneering AI company and the vibrant community hub building the future of machine learning (ML). It serves as the leading collaboration platform where machine learning engineers, scientists, researchers, and enthusiasts come together to create, share, and advance open-source ML models, datasets, and applications. Focused on democratizing machine learning, Hugging Face empowers the next generation of AI practitioners through an open, ethical, and inclusive AI ecosystem.

---

## What We Offer

- **Model Hub:** Access and contribute to over 2 million machine learning models across modalities including text, image, video, audio, and 3D.
- **Datasets:** Browse and use from a growing collection of 500,000+ datasets essential for training and benchmarking ML models.
- **Spaces:** Host and explore ML-powered applications and demo projects built by the community.
- **Open Source Stack:** Leverage an industry-leading suite of open-source libraries and tools designed to accelerate your ML workflows.
- **Enterprise Solutions:** Tailored offerings for businesses wanting to integrate cutting-edge AI capabilities.

---

## Our Community and Customers

Hugging Face is the beating heart of the AI revolution, trusted and actively used by over 80,000 machine learning enthusiasts and professionals globally. Our community continuously contributes to shared datasets, constantly updates models, and develops novel AI applications. Our user base ranges from individual developers and researchers to large enterprises adopting AI-driven innovation. The platform fosters collaboration, transparency, and learning, ensuring both novices and experts can build their portfolios and accelerate their AI skills.

---

## Our Culture and Mission

We believe in democratizing *“good machine learning, one commit at a time.”* Our culture is built on open collaboration, ethical AI development, and community empowerment. Hugging Face fosters a welcoming, inclusive environment where innovation thrives through sharing and co-creation. Our talented interdisciplinary team is devoted to pushing the boundaries of AI while maintaining a responsible and human-centered approach.

---

## Careers at Hugging Face

Joining Hugging Face means being part of a fast-growing, dynamic team passionate about open-source AI. We are constantly looking for:

- Machine Learning Engineers
- Research Scientists
- Software Developers
- Community Managers
- Product Designers

If you want to contribute to shaping the future of AI and share a commitment to openness, transparency, and ethics, Hugging Face welcomes you to join our mission-driven team. Discover current opportunities and join us in building the AI community of tomorrow.

---

## Connect with Us

- Website: [huggingface.co](https://huggingface.co)
- GitHub, Twitter, LinkedIn, Discord communities  
- Access documentation, blogs, and forums to learn and engage with the latest AI content

---

**Hugging Face** – The AI community building the future. Join us to create, explore, and shape the future of machine learning together!

In [32]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


# Hugging Face: Your Friendly Neighborhood AI Community 🤗

Welcome to **Hugging Face**, the AI community that’s not just building the future — they’re *hugging* it into existence with open arms (and open source, of course)!

---

## Why Hugging Face? Because Building AI Should Be a Team Sport

Imagine a place where machine learning engineers, scientists, developers, and dreamers come together to:

- **Share** over 2 million models and 500,000+ datasets
- **Collaborate** on projects spanning text, images, video, audio, and even 3D (yes, 3D! Welcome to the sci-fi future)
- **Explore** and experiment in public spaces with countless applications
- **Accelerate** your machine learning journey with a robust, open source toolkit  
 
All this, wrapped in a warm, inviting community vibe — no tech jargon or robot overlords, just people powering the AI revolution together.

---

## The Hugging Face Hub: Like GitHub, But For AI Wizards 🧙‍♂️✨

Here’s what makes Hugging Face the go-to spot for AI folk worldwide:

- **Models Galore:** Browse millions of ready-to-use models – from image generation to advanced reasoning. No need to reinvent the wheel, just start spinning!
- **Datasets That Don’t Mess Around:** Explore, share, and contribute to huge datasets with ease — because AI eats data for breakfast.
- **Spaces:** Run and showcase your AI apps live. It’s your personal AI playground, ready to impress your friends or future employers.
- **Open & Ethical:** Built on open-source principles with a strong commitment to building a responsible AI future. Robots may rise, but ethics come first.

---

## Enterprise Plans: For When You Want to Hug Hugging Face *Even More*

Got a team? A whole company? Fancy some security, privacy, and scalability without breaking a sweat?

- Team plans start at $20/user/month — affordable AI hugs for all your teammates.  
- Enterprise options come with **Single Sign-On**, **granular access controls**, **advanced analytics**, and even more **compute power** (hello, ZeroGPU boosts!).  
- Keep your private projects private, and your data safe with dedicated storage and audit logs.  
- Scale your AI efforts securely across regions, with custom contract options tailored to your needs.

---

## Life at Hugging Face: Work Hard, Hug Harder 🤗💼

At Hugging Face, it’s not just about AI — it’s about *people*. The culture here is as friendly as the name suggests:

- **Collaborative & Inclusive:** Bring your best ideas, get instant feedback, and grow alongside passionate AI enthusiasts.
- **Innovation on Steroids:** Work with some of the brightest minds pushing tech boundaries every day.
- **Open Source Spirit:** Your work contributes to the community, helping shape the AI landscape worldwide.
- **Remote & Flexible:** Because hugs come in many forms — whether in-person or through your screen.

If you’re a machine learning engineer, data scientist, developer, or just a curious innovator, Hugging Face wants *you* to join the party.

👉 Check out the [careers page](https://huggingface.co/careers) and become part of the AI family!

---

## Customers? Oh, Just Everyone Feeding the AI Revolution

From startups building the next cool bot, to Fortune 500 companies securing enterprise AI, Hugging Face is the backbone powering the machine learning community at large. Universities, researchers, hobbyists — all united by one thing: building better AI, together.

---

## Brand Colors & Vibes

- A sunny **#FFD21E** yellow that feels like optimism (and sunshine on your code)
- A warm **#FF9D00** orange to spark creativity and innovation
- A grounded **#6B7280** gray to keep things professional yet friendly

It's all about sunshine, warmth, and tech savvy — just like the team!

---

## Ready to Hug the Future?

Join **Hugging Face** today. Whether you want to share your latest neural network, find the perfect dataset, or just hang out with the friendliest AI community on the web — come for the models, stay for the hugs.

**Sign up now and give your AI projects the warmest welcome!**

---

*Hugging Face — where machine learning meets human connection.*  
*(And yes, there’s a smiling face emoji on purpose.)* 🤗

---

Find out more:  
- Explore the Hub: [huggingface.co](https://huggingface.co)  
- Enterprise solutions: [huggingface.co/enterprise](https://huggingface.co/enterprise)  
- Careers & culture: [huggingface.co/careers](https://huggingface.co/careers)  
- Follow the fun: [GitHub](https://github.com/huggingface), [Twitter](https://twitter.com/huggingface), [Discord](https://discord.gg/huggingface)

---

*Disclaimer: Hugging Face does not offer actual hugs... but the community comes pretty close!*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>